# TokenScope — Interactive Results Analysis

This notebook provides an interactive interface for exploring benchmark results.
Run the cells sequentially after executing benchmarks.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from analysis.figure_style import apply_style
from analysis.load_results import load_aggregate_csv, group_by, extract_series

apply_style()
%matplotlib inline

RESULTS_DIR = '../results'

In [ ]:
# Load aggregate results
rows = load_aggregate_csv(RESULTS_DIR)
print(f'Loaded {len(rows)} result rows')

if rows:
    # Show available columns
    print('Columns:', list(rows[0].keys()))
    
    # Unique configurations
    models = set(r.get('model_id', '') for r in rows)
    devices = set(r.get('device', '') for r in rows)
    print(f'Models: {models}')
    print(f'Devices: {devices}')

In [ ]:
# Per-token latency vs prompt length
if rows:
    groups = group_by(rows, 'model_id')
    fig, ax = plt.subplots(figsize=(9, 6))
    for label, group_rows in groups.items():
        x, y = extract_series(group_rows, 'prompt_length', 'per_token_mean_ms')
        if len(x) > 0:
            ax.plot(x, y, 'o-', label=str(label))
    ax.set_xlabel('Prompt Length (tokens)')
    ax.set_ylabel('Per-token Latency (ms)')
    ax.set_title('Decode Latency Scaling')
    ax.legend()
    plt.show()

In [ ]:
# KV cache model visualization
from analysis.kv_cache_model import KNOWN_ARCHITECTURES, kv_cache_curve, CACHE_SIZES

fig, ax = plt.subplots(figsize=(10, 6))
for name in ['llama-7b', 'llama2-70b', 'llama3-8b']:
    arch = KNOWN_ARCHITECTURES[name]
    seq, kv_mb = kv_cache_curve(arch, max_seq=8192)
    ax.plot(seq, kv_mb, label=arch.name)

for label, size in list(CACHE_SIZES.items())[:3]:
    ax.axhline(size / (1024**2), color='gray', linestyle=':', alpha=0.4)
    ax.text(100, size / (1024**2), label, fontsize=7, color='gray')

ax.set_xlabel('Sequence Length')
ax.set_ylabel('KV Cache (MB)')
ax.set_title('KV Cache Size Scaling')
ax.legend()
plt.show()